In [0]:
from pyspark.sql import functions as F
import time

base = "/Volumes/workspace/default/m5/"
t0 = time.time()
sales  = spark.read.csv(base + "sales_train_evaluation.csv", header=True, inferSchema=True)
print(sales.count(), "rows —", f"{time.time()-t0:.1f}s")
cal    = spark.read.csv(base + "calendar.csv", header=True, inferSchema=True)
prices = spark.read.csv(base + "sell_prices.csv", header=True, inferSchema=True)

print("sales :", sales.count(), "rows ×", len(sales.columns), "cols")
print("cal   :", cal.count())
print("prices:", prices.count())

sales.select("id", "item_id", "store_id", "d_1", "d_2", "d_1941").show(3)

30490 rows — 3.5s
sales : 30490 rows × 1947 cols
cal   : 1969
prices: 6841121
+--------------------+-------------+--------+---+---+------+
|                  id|      item_id|store_id|d_1|d_2|d_1941|
+--------------------+-------------+--------+---+---+------+
|HOBBIES_1_001_CA_...|HOBBIES_1_001|    CA_1|  0|  0|     1|
|HOBBIES_1_002_CA_...|HOBBIES_1_002|    CA_1|  0|  0|     0|
|HOBBIES_1_003_CA_...|HOBBIES_1_003|    CA_1|  0|  0|     1|
+--------------------+-------------+--------+---+---+------+
only showing top 3 rows


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import time

id_cols  = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [f"d_{i}" for i in range(1, 1942)]   # d_1 … d_1941

schema = StructType(
    [StructField(c, StringType(), False) for c in id_cols] +
    [StructField(c, IntegerType(), True) for c in day_cols]
)

t0 = time.time()
sales = spark.read.csv(base + "sales_train_evaluation.csv", header=True, schema=schema)
print(sales.count(), "rows —", f"{time.time()-t0:.1f}s")

30490 rows — 1.2s


In [0]:
# ---- Reshape wide → long ---------------------------------------------------
# M5 ships one row per item-store with 1,941 day columns (d_1 … d_1941).
# Downstream steps (calendar/price joins, window features, partitioning)
# all need one row per item-store-day, so we unpivot here.
#
# Spark has no DataFrame.melt(); the idiomatic way is the SQL `stack()` generator:
#   stack(n, label_1, value_1, label_2, value_2, ...) AS (label_col, value_col)
# It expands n (label, value) pairs into n rows. With 1,941 pairs we build the
# expression string programmatically rather than typing it out.
#
# Cost note: stack() is a narrow transformation — each input row expands
# independently, no shuffle. The expensive step is the partitioned write below.

stack_expr = "stack({n}, {pairs}) as (d, sales)".format(
    n=len(day_cols),
    pairs=", ".join(f"'{c}', `{c}`" for c in day_cols),   # 'd_1', `d_1`, 'd_2', `d_2`, ...
)

# Keep the six id columns; every output row carries its item/store identifiers.
long = sales.select(*id_cols, F.expr(stack_expr))

# stack() is lazy — count() forces execution. Expect 30,490 × 1,941 = 59,181,090 rows.
t0 = time.time()
n = long.count()
print(f"{n:,} rows — {time.time()-t0:.1f}s")
long.show(5)

59,181,090 rows — 5.3s
+--------------------+-------------+---------+-------+--------+--------+---+-----+
|                  id|      item_id|  dept_id| cat_id|store_id|state_id|  d|sales|
+--------------------+-------------+---------+-------+--------+--------+---+-----+
|HOBBIES_1_001_CA_...|HOBBIES_1_001|HOBBIES_1|HOBBIES|    CA_1|      CA|d_1|    0|
|HOBBIES_1_002_CA_...|HOBBIES_1_002|HOBBIES_1|HOBBIES|    CA_1|      CA|d_1|    0|
|HOBBIES_1_003_CA_...|HOBBIES_1_003|HOBBIES_1|HOBBIES|    CA_1|      CA|d_1|    0|
|HOBBIES_1_004_CA_...|HOBBIES_1_004|HOBBIES_1|HOBBIES|    CA_1|      CA|d_1|    0|
|HOBBIES_1_005_CA_...|HOBBIES_1_005|HOBBIES_1|HOBBIES|    CA_1|      CA|d_1|    0|
+--------------------+-------------+---------+-------+--------+--------+---+-----+
only showing top 5 rows


In [0]:
# ---- Persist as Parquet, partitioned by store_id ---------------------------
# 10 stores → 10 partition directories (store_id=CA_1 … store_id=WI_3).
# Any downstream query filtering on store_id reads 1/10 of the data
# (partition pruning). Row counts per store are near-identical (~5.9M),
# so this key gives balanced partitions with no skew.
#
# Two variants are written so the file-layout trade-off is measurable:
#   A) partitionBy only  → each store dir gets one file per Spark partition
#      that holds rows for that store (here 1–2, since the CSV is store-ordered)
#   B) repartition("store_id") first → one Spark partition per store,
#      so exactly one file per directory. Costs one extra shuffle at write.

def count_files(path):
    """Data files per store_id= directory (ignores _SUCCESS / _started_ / _committed_ markers)."""
    return {
        p.name.rstrip("/"): sum(1 for f in dbutils.fs.ls(p.path) if f.name.endswith(".parquet"))
        for p in dbutils.fs.ls(path)
        if p.name.startswith("store_id=")
    }

n_parts = long.select(F.spark_partition_id().alias("pid")).distinct().count()
print("Spark partitions in `long`:", n_parts)

# ---- Variant A: partitionBy only -------------------------------------------
out_a = base + "out/sales_long_a"
t0 = time.time()
long.write.mode("overwrite").partitionBy("store_id").parquet(out_a)
t_a = time.time() - t0
print(f"A  write: {t_a:.1f}s   files/dir: {count_files(out_a)}")

# ---- Variant B: repartition("store_id") then partitionBy --------------------
out = base + "out/sales_long"          # canonical output
t0 = time.time()
long.repartition("store_id").write.mode("overwrite").partitionBy("store_id").parquet(out)
t_b = time.time() - t0
print(f"B  write: {t_b:.1f}s   files/dir: {count_files(out)}")

# ---- Verify round-trip -----------------------------------------------------
back = spark.read.parquet(out)
assert back.count() == n, "row count changed on write/read"
print("round-trip OK:", f"{back.count():,}", "rows")
back.groupBy("store_id").count().orderBy("store_id").show()   # 5,918,109 each, no skew

# A was only for measurement — keep one canonical copy.
dbutils.fs.rm(out_a, recurse=True)

Spark partitions in `long`: 8
A  write: 6.9s   files/dir: {'store_id=CA_1': 1, 'store_id=CA_2': 2, 'store_id=CA_3': 2, 'store_id=CA_4': 2, 'store_id=TX_1': 1, 'store_id=TX_2': 2, 'store_id=TX_3': 2, 'store_id=WI_1': 2, 'store_id=WI_2': 2, 'store_id=WI_3': 1}
B  write: 8.8s   files/dir: {'store_id=CA_1': 1, 'store_id=CA_2': 1, 'store_id=CA_3': 1, 'store_id=CA_4': 1, 'store_id=TX_1': 1, 'store_id=TX_2': 1, 'store_id=TX_3': 1, 'store_id=WI_1': 1, 'store_id=WI_2': 1, 'store_id=WI_3': 1}
round-trip OK: 59,181,090 rows
+--------+-------+
|store_id|  count|
+--------+-------+
|    CA_1|5918109|
|    CA_2|5918109|
|    CA_3|5918109|
|    CA_4|5918109|
|    TX_1|5918109|
|    TX_2|5918109|
|    TX_3|5918109|
|    WI_1|5918109|
|    WI_2|5918109|
|    WI_3|5918109|
+--------+-------+



True